## Set-up RaspberryPi

To set-up the RaspberryPi for this project, you can follow this short documentation.

You need a RaspberryPi, a laptop and a USB stick to set up the RaspberryPi. 

1. Download the Raspberry Pi Imager
2. Plug the USB drive into your PC
3. Set up the Imager on the USB drive according to the instructions
	- Set up a hotspot connection
4. Check if the “ssh” file is on the USB drive!
5. Remove the USB drive
6. Plug the USB drive into the RaspberryPi
7. Wait a moment
8. In PowerShell, enter “ssh username@hostname.local”
9. The output should turn green at the end
10. In VS Code, download the “Remote - SSH” extension
11. Ctrl Shift P -> Remote-SSH: Open SSH Configuration File…
12. Enter the following information: 
	- Host
	- Hostname
	- Username
	- Specify port
13. Save the configuration file

### Connect to RaspberryPi in Visual Studio Code via SSH
To open the RaspberryPi in Visual Studio code on your laptop via SSH, you need to follow these steps:

1. Open VS Code
2. Press Ctrl+Shift+P and select “Remote SSH: Connect to Host”
3. A new window will open; enter your password in the field at the top
4. Activate the virtual environment in the terminal with the command: source ~/venv/bin/activate
5. You can now use Linux commands in this window

→ Scripts are automatically saved here
→ You need to be connected to the same WLAN / Hotspot which is safed on the RaspberryPi

### Set-up RaspberryPi and Dobot

The basic workflow is: VS Code → SSH → RaspberryPi → Python script → Dobot Magician

1. Check the connection between the RaspberryPi and the Dobot
	- Connect the Dobot to the RaspberryPi via USB and check the following in the RaspberryPi’s terminal:
	    - ls /dev/ttyUSB* → e.g. /dev/ttyUSB0 or ls /dev/ttyACM* → e.g. /dev/ttyACM0
		- This is the serial port used to communicate with the robot.
2. Install the Dobot API on the RaspberryPi
	- sudo apt updatesudo apt install python3-pippip3 install pyserial
	- clone the SDK: git clone https://github.com/Dobot-Arm/DobotDemoForPython.gitcd DobotDemoForPython
3. Try a sample script to move the Dobot (some sample code is provided below)
4. Run the script from VS Code
	- If you’re connected to VS Code via SSH (Remote SSH Extension) and your virtual environment is activated, you can run the script directly: python3 dobot_test.py

Common issues
- Permission error
	- If /dev/ttyUSB0 is inaccessible:
		- sudo usermod -a -G dialout $USER
		- Then restart the RaspberryPi
- Check the port
	- dmesg | grep tty
    - shows which USB port was detected.

In [ ]:
# sample code

from dobotapi import Dobot
import time

# wait for Dobot to be ready
time.sleep(0.2)

port = "COM7" # <-- adjust this to your Dobot's port
device = Dobot(port=port)
device.connect()

home = (200.0, 200.0, 80.0, 25.0) 

# start conveyor belt
device.conveyor_belt.move(speed=0.5)  
print("✓ conveyor belt started")

# waite for 2 seconds to let the object reach the sensor
time.sleep(2)

# wait until object is detected by IR sensor
while device.get_ir() == False:
    time.sleep(0.1)

print("🔴 Object detected - STOP")
device.conveyor_belt.idle()

# Dobot to home position
print("🏠 drive to home...")
device.move_to(*home)
time.sleep(0.3)

# Gripper open
print("🖐️  Gripper opened...")
device.gripper.open()
time.sleep(0.3)
print("✓ Ready for next object")

# Gripper close
device.gripper.close()
print("✓ Gripper closed")
time.sleep(2)

pos = device.get_pose()

print("X:", pos[0])
print("Y:", pos[1])
print("Z:", pos[2])
print("R:", pos[3])

# Stop conveyor belt
device.conveyor_belt.idle()
print("✓ Conveyor belt stopped")

# Close connection
device.close()
print("✓ Finished")